In [33]:
# PCA 
# Example of loading your dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# Load your data
data = pd.read_csv('train.csv')

# Check the columns
#print(data.columns)

# Ensure the target column name is correct
target_column_name = 'fake'  # Replace with actual name after checking

# Split the features and target
X = data.drop(target_column_name, axis=1)
y = data[target_column_name]

# Split data into training and testing datasets
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Now you can use x_train in your standardization function
def standardize_data(X):
    mean = np.mean(X, axis=0)
    std_dev = np.std(X, axis=0)
    X_scaled = (X - mean) / std_dev
    return X_scaled, mean, std_dev

X_train_scaled, mean_train, std_dev_train = standardize_data(x_train)
X_test_scaled = (x_test - mean_train) / std_dev_train

cov_matrix = np.cov(X_train_scaled.T)

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

k = 5 
top_indices = np.argsort(eigenvalues)[::-1][:k]
top_eigenvectors = eigenvectors[:, top_indices]

X_train_pca = X_train_scaled.dot(top_eigenvectors)
X_test_pca = X_test_scaled.dot(top_eigenvectors)

naive_bayes = GaussianNB()
naive_bayes.fit(X_train_pca, y_train)

y_pred = naive_bayes.predict(X_test_pca)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy in naive bayes:", accuracy)


Accuracy in naive bayes: 0.8448275862068966


In [34]:
#decision tree
from sklearn.tree import DecisionTreeClassifier

k = 10 
accuracy_scores = []
fold_size = len(X_train_pca) // k
for i in range(k):
    start_idx = i * fold_size
    end_idx = start_idx + fold_size
    X_val_fold = X_train_pca[start_idx:end_idx]
    y_val_fold = y_train[start_idx:end_idx]
    
    X_train_fold = np.concatenate([X_train_pca[:start_idx], X_train_pca[end_idx:]], axis=0)
    y_train_fold = np.concatenate([y_train[:start_idx], y_train[end_idx:]], axis=0)
    
    decision_tree = DecisionTreeClassifier()
    decision_tree.fit(X_train_fold, y_train_fold)

    y_pred_fold = decision_tree.predict(X_val_fold)

    accuracy_fold = accuracy_score(y_val_fold, y_pred_fold)
    accuracy_scores.append(accuracy_fold)

average_accuracy = np.mean(accuracy_scores)
print("Average Accuracy:", average_accuracy)


Average Accuracy: 0.9065217391304348


In [35]:
# Random Forest
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

random_forest = RandomForestClassifier()
random_forest.fit(X_train_pca, y_train)
rf_y_pred = random_forest.predict(X_test_pca)
rf_accuracy = accuracy_score(y_test, rf_y_pred)
print("Random Forest Accuracy:", rf_accuracy)
# Gradient Boosting
gradient_boosting = GradientBoostingClassifier()
gradient_boosting.fit(X_train_pca, y_train)
gb_y_pred = gradient_boosting.predict(X_test_pca)
gb_accuracy = accuracy_score(y_test, gb_y_pred)
print("Gradient Boosting Accuracy:", gb_accuracy)

Random Forest Accuracy: 0.896551724137931
Gradient Boosting Accuracy: 0.9224137931034483


In [36]:
from sklearn.model_selection import GridSearchCV
from joblib import dump

gb_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}

gb_grid_search = GridSearchCV(GradientBoostingClassifier(), param_grid=gb_param_grid, cv=5)
gb_grid_search.fit(x_train, y_train)

best_gb_model = gb_grid_search.best_estimator_

dump(best_gb_model, 'best_gradient_boosting_model.joblib')

gb_y_pred = best_gb_model.predict(x_train)
gb_accuracy = accuracy_score(y_train, gb_y_pred)
print("Gradient Boosting Accuracy using GridSearchCV:", gb_accuracy)

Gradient Boosting Accuracy using GridSearchCV: 1.0


In [37]:
from joblib import load
import pandas as pd

best_gb_model = load('best_gradient_boosting_model.joblib')
new_data = pd.read_csv(r'profile_info.csv')  # Replace 'your_data_for_prediction.csv' with the path to your CSV file


X_new = new_data
predictions = best_gb_model.predict(X_new)
print("Predictions:")
for pred in predictions:
    print("Fake" if pred == 0 else "Not Fake")

Predictions:
Not Fake
Not Fake
Not Fake
Fake
